# Part II: Core Concepts - Code Examples

This notebook covers Chapters 5-8:
- **Chapter 5**: Measurement — forcing diagonality
- **Chapter 6**: Composite Systems — tensor products
- **Chapter 7**: Entanglement — non-factorizable states
- **Chapter 8**: Decoherence — quantum → classical transition

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

## Helper Functions

In [ ]:
def tensor_product(A, B):
    """Tensor (Kronecker) product."""
    return np.kron(A, B)


def partial_trace_B(rho_AB, dim_A, dim_B):
    """Trace out system B from composite density matrix."""
    rho_A = np.zeros((dim_A, dim_A), dtype=complex)
    for j in range(dim_B):
        proj = np.zeros((dim_B, 1))
        proj[j] = 1
        operator = tensor_product(np.eye(dim_A), proj.T)
        rho_A += operator @ rho_AB @ operator.conj().T
    return rho_A


def von_neumann_entropy(rho):
    """Von Neumann entropy in bits."""
    eigenvalues = np.linalg.eigvalsh(rho)
    eigenvalues = eigenvalues[eigenvalues > 1e-10]
    if len(eigenvalues) == 0:
        return 0.0
    return -np.sum(eigenvalues * np.log2(eigenvalues))


def purity(rho):
    """Tr(ρ²)"""
    return np.real(np.trace(rho @ rho))


# Basis states
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)
ket_plus = (ket_0 + ket_1) / np.sqrt(2)
ket_minus = (ket_0 - ket_1) / np.sqrt(2)

---
## Chapter 5: Measurement

Measurement kills off-diagonals → state becomes classical.

In [ ]:
def measure_in_basis(rho, basis_states):
    """Apply projective measurement, return post-measurement state and probabilities."""
    probs = []
    rho_post = np.zeros_like(rho)

    for state in basis_states:
        state = np.array(state, dtype=complex)
        state = state / np.linalg.norm(state)
        P = np.outer(state, np.conj(state))
        prob = np.real(np.trace(P @ rho))
        probs.append(prob)
        rho_post += P @ rho @ P

    return rho_post, np.array(probs)


# Superposition state
psi = ket_plus
rho = np.outer(psi, np.conj(psi))

print("=== Measurement in Computational Basis ===")
print(f"Before: ρ =\n{rho}")
print(f"Entropy: {von_neumann_entropy(rho):.3f} bits (pure state)")

rho_after, probs = measure_in_basis(rho, [ket_0, ket_1])
print(f"\nAfter measurement in {{|0⟩, |1⟩}}:")
print(f"ρ =\n{rho_after}")
print(f"Probabilities: {probs}")
print(f"Entropy: {von_neumann_entropy(rho_after):.3f} bits (mixed state!)")

=== Measurement in Computational Basis ===
Before: ρ =
[[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]
Entropy: 0.000 bits (pure state)

After measurement in {|0⟩, |1⟩}:
ρ =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
Probabilities: [0.5 0.5]
Entropy: 1.000 bits (mixed state!)


In [4]:
# Measurement in Hadamard basis — |+⟩ is an eigenstate!
print("=== Measurement in Hadamard Basis ===")
print(f"State |+⟩ measured in {{|+⟩, |−⟩}} basis:")

rho_after_h, probs_h = measure_in_basis(rho, [ket_plus, ket_minus])
print(f"Probabilities: {probs_h}")
print(f"Entropy after: {von_neumann_entropy(rho_after_h):.3f} bits")
print("→ No entropy created! Measured in the 'right' basis.")

=== Measurement in Hadamard Basis ===
State |+⟩ measured in {|+⟩, |−⟩} basis:
Probabilities: [1. 0.]
Entropy after: -0.000 bits
→ No entropy created! Measured in the 'right' basis.


---
## Chapter 6: Composite Systems

Tensor products: dimensions multiply, but some states can't be factored.

In [5]:
# Two-qubit basis
ket_00 = tensor_product(ket_0, ket_0)
ket_01 = tensor_product(ket_0, ket_1)
ket_10 = tensor_product(ket_1, ket_0)
ket_11 = tensor_product(ket_1, ket_1)

print("=== Tensor Product Basis ===")
print(f"|00⟩ = {ket_00}")
print(f"|01⟩ = {ket_01}")
print(f"|10⟩ = {ket_10}")
print(f"|11⟩ = {ket_11}")
print(f"\nDimension: 2 × 2 = {len(ket_00)}")

=== Tensor Product Basis ===
|00⟩ = [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
|01⟩ = [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
|10⟩ = [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
|11⟩ = [0.+0.j 0.+0.j 0.+0.j 1.+0.j]

Dimension: 2 × 2 = 4


In [6]:
# Product state: |+⟩ ⊗ |0⟩
psi_product = tensor_product(ket_plus, ket_0)
rho_product = np.outer(psi_product, np.conj(psi_product))

print("=== Product State |+⟩ ⊗ |0⟩ ===")
print(f"ρ_AB =\n{rho_product.real}")

# Partial trace
rho_A = partial_trace_B(rho_product, 2, 2)
print(f"\nρ_A (tracing out B) =\n{rho_A}")
print(f"Entropy of A: {von_neumann_entropy(rho_A):.3f} bits")

=== Product State |+⟩ ⊗ |0⟩ ===
ρ_AB =
[[0.5 0.  0.5 0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.5 0. ]
 [0.  0.  0.  0. ]]

ρ_A (tracing out B) =
[[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]
Entropy of A: 0.000 bits


---
## Chapter 7: Entanglement

Non-factorizable states: pure global state → mixed local states!

In [7]:
# Bell state |Φ+⟩ = (|00⟩ + |11⟩)/√2
phi_plus = (ket_00 + ket_11) / np.sqrt(2)
rho_bell = np.outer(phi_plus, np.conj(phi_plus))

print("=== Bell State |Φ+⟩ = (|00⟩ + |11⟩)/√2 ===")
print(f"Global ρ_AB =\n{rho_bell.real}")
print(f"Global purity: {purity(rho_bell):.3f} (pure!)")
print(f"Global entropy: {von_neumann_entropy(rho_bell):.3f} bits")

# Partial trace
rho_A_bell = partial_trace_B(rho_bell, 2, 2)
print(f"\nLocal ρ_A =\n{rho_A_bell}")
print(f"Local purity: {purity(rho_A_bell):.3f} (maximally mixed!)")
print(f"Local entropy: {von_neumann_entropy(rho_A_bell):.3f} bits")
print("\n→ Pure global state, but maximally uncertain locally. This is ENTANGLEMENT.")

=== Bell State |Φ+⟩ = (|00⟩ + |11⟩)/√2 ===
Global ρ_AB =
[[0.5 0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0.5]]
Global purity: 1.000 (pure!)
Global entropy: 0.000 bits

Local ρ_A =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
Local purity: 0.500 (maximally mixed!)
Local entropy: 1.000 bits

→ Pure global state, but maximally uncertain locally. This is ENTANGLEMENT.


In [ ]:
def entropy_of_entanglement(psi_AB, dim_A, dim_B):
    """Entanglement entropy = S(ρ_A) for pure bipartite state."""
    rho_AB = np.outer(psi_AB, np.conj(psi_AB))
    rho_A = partial_trace_B(rho_AB, dim_A, dim_B)
    return von_neumann_entropy(rho_A)


print("=== Entanglement vs Angle ===")
print("State: |ψ⟩ = cos(θ)|00⟩ + sin(θ)|11⟩")
print(f"{'θ (deg)':<10} {'Entanglement (bits)':<20}")
print("-" * 30)

for theta_deg in [0, 15, 30, 45]:
    theta = np.radians(theta_deg)
    psi = np.cos(theta) * ket_00 + np.sin(theta) * ket_11
    E = entropy_of_entanglement(psi, 2, 2)
    print(f"{theta_deg:<10} {E:<20.3f}")

print("\nθ=0°: product state (no entanglement)")
print("θ=45°: maximally entangled (1 ebit)")

=== Entanglement vs Angle ===
State: |ψ⟩ = cos(θ)|00⟩ + sin(θ)|11⟩
θ (deg)    Entanglement (bits) 
------------------------------
0          -0.000              
15         0.355               
30         0.811               
45         1.000               

θ=0°: product state (no entanglement)
θ=45°: maximally entangled (1 ebit)


---
## Chapter 8: Decoherence

Environment interaction kills coherence → quantum becomes classical.

In [ ]:
def dephasing_channel(rho, gamma):
    """Dephasing channel: γ=0 → no change, γ=1 → fully diagonal."""
    diagonal = np.diag(np.diag(rho))
    return (1 - gamma) * rho + gamma * diagonal


# Superposition undergoing decoherence
psi = ket_plus
rho_0 = np.outer(psi, np.conj(psi))

print("=== Decoherence Trajectory ===")
print(f"{'γ':<8} {'Off-diag':<12} {'Entropy':<12} {'Purity':<12}")
print("-" * 45)

for gamma in [0.0, 0.25, 0.5, 0.75, 1.0]:
    rho = dephasing_channel(rho_0, gamma)
    S = von_neumann_entropy(rho)
    P = purity(rho)
    off_diag = rho[0, 1].real
    print(f"{gamma:<8.2f} {off_diag:<12.3f} {S:<12.3f} {P:<12.3f}")

=== Decoherence Trajectory ===
γ        Off-diag     Entropy      Purity      
---------------------------------------------
0.00     0.500        0.000        1.000       
0.25     0.375        0.544        0.781       
0.50     0.250        0.811        0.625       
0.75     0.125        0.954        0.531       
1.00     0.000        1.000        0.500       


In [ ]:
# Quantum advantage under decoherence
print("=== Quantum Advantage Under Decoherence ===")
print("(Perturbed coin example)")

p = 0.3
s0 = np.array([np.sqrt(1 - p), np.sqrt(p)])
s1 = np.array([np.sqrt(p), np.sqrt(1 - p)])
rho_q = 0.5 * np.outer(s0, s0) + 0.5 * np.outer(s1, s1)

C_mu = 1.0  # Classical complexity

print(f"\n{'γ':<8} {'C_q':<12} {'C_μ':<12} {'Advantage':<12}")
print("-" * 45)

for gamma in [0.0, 0.25, 0.5, 0.75, 1.0]:
    rho = dephasing_channel(rho_q, gamma)
    C_q = von_neumann_entropy(rho)
    advantage = C_mu - C_q
    print(f"{gamma:<8.2f} {C_q:<12.3f} {C_mu:<12.3f} {advantage:<12.3f}")

print("\n→ Decoherence destroys quantum advantage!")

=== Quantum Advantage Under Decoherence ===
(Perturbed coin example)

γ        C_q          C_μ          Advantage   
---------------------------------------------
0.00     0.250        1.000        0.750       
0.25     0.625        1.000        0.375       
0.50     0.843        1.000        0.157       
0.75     0.962        1.000        0.038       
1.00     1.000        1.000        0.000       

→ Decoherence destroys quantum advantage!


---
## Key Takeaways

1. **Measurement** = projection → off-diagonals killed → entropy increases
2. **Tensor products** = composite systems → dimensions multiply
3. **Entanglement** = non-factorizable → pure global, mixed local
4. **Decoherence** = environment entanglement → coherence lost → classical emerges

The thread: **off-diagonals are quantum, diagonals are classical**.